# Submission: lgbm_building, 31 leaves

Report: [baseline_report.md](../../baseline_report.md#submission)


In [1]:
import os
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.metrics import rmsle
from ashrae.submission import build_submission, check_submission, to_readings
from ashrae.tracking import log_run

SMOKE = os.environ.get("SMOKE") == "1"
CHUNK_SIZE = 1_000_000  # test rows predicted at once
if SMOKE:
    EXPERIMENT = "ashrae-submission-smoke"
    N_TRAIN = 20_000
    N_TEST = 50_000
    LGBM_PARAMS = {"n_estimators": 20, "learning_rate": 0.1, "num_leaves": 15}
    SUBMISSION_PATH = None  # partial test set, so nothing to submit
else:
    EXPERIMENT = "ashrae-submission"
    N_TRAIN = 2_000_000  # the same rows as the blend in submission.ipynb
    N_TEST = None  # all 41.7M test rows
    LGBM_PARAMS = {
        "n_estimators": 1000,
        "learning_rate": 0.1,
        "num_leaves": 31,
    }  # exp06: same gap as untuned (about 0.14) with shallower trees
    SUBMISSION_PATH = PROCESSED / "submission_lgbm_building_31_leaves.csv"

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Training rows


In [2]:
df = read_processed()
BUILDING = pd.CategoricalDtype(
    range(int(df.building_id.max()) + 1)
)  # all ids, so train and test share categories
train_df = df[~df.anomaly].sample(
    N_TRAIN, random_state=0
)  # all of 2016, anomalies dropped
del df

year_median = train_df.year_built.median()  # reused for the test set
X_train = build_features(train_df, year_median)
y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features + building_id")


def with_building(X, rows):
    """`X` plus `building_id` as a categorical, which LightGBM splits on directly."""
    building = rows.building_id.astype(int).astype(BUILDING)
    return X.assign(building_id=building.to_numpy())

2,000,000 training rows, 33 features + building_id


## Model


In [3]:
X_train_lgbm = with_building(X_train, train_df)
lgbm = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1).fit(
    X_train_lgbm, y_train
)
train_rmsle = rmsle(y_train, lgbm.predict(X_train_lgbm))

log_run(
    EXPERIMENT,
    f"lgbm_building_31_leaves | {N_TRAIN:,} rows | submission",
    params={**LGBM_PARAMS, "n_train": N_TRAIN},
    metrics={"train_rmsle": train_rmsle},
    tags={"model": "lgbm_building_31_leaves", "notebook": "submission_31_leaves"},
)

print(f"lgbm_building_31_leaves: training RMSLE {train_rmsle:.3f}")

lgbm_building_31_leaves: training RMSLE 0.844


## Test predictions


In [4]:
test_df = read_processed("test_merged.parquet")
if N_TEST:
    test_df = test_df.head(N_TEST)


def predict_log(rows):
    """Clipped log1p predictions for `rows`, built with the same feature columns as training."""
    X = build_features(rows, year_median).reindex(columns=X_train.columns, fill_value=0)
    return np.clip(lgbm.predict(with_building(X, rows)), 0, None)


starts = range(0, len(test_df), CHUNK_SIZE)
lgbm_log = np.concatenate(
    [predict_log(test_df.iloc[start : start + CHUNK_SIZE]) for start in starts]
)
print(f"mean log1p prediction: {lgbm_log.mean():.3f}")

mean log1p prediction: 4.234


In [ ]:
reading = to_readings(lgbm_log, test_df)  # site 0 electricity back in kBTU
submission = build_submission(test_df, reading)
if SUBMISSION_PATH:
    check_submission(submission)
    submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.4f")
    print(f"saved {SUBMISSION_PATH}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/data/processed/submission_lgbm_building_31_leaves.csv: 41,697,600 rows


In [ ]:
if SUBMISSION_PATH:  # multithreaded 7-Zip CLI (brew install sevenzip), much faster than py7zr
    !cd "{SUBMISSION_PATH.parent}" && rm -f "{SUBMISSION_PATH.stem}.7z" && 7zz a -mmt=on "{SUBMISSION_PATH.stem}.7z" "{SUBMISSION_PATH.name}"